In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ---- CONFIG ----
PROJECT = Path("/home/fs01/jl2966/acorn-julia")  # repo root
RUN_NAME = "low_RE_high_elec_iter0"
EXPERIMENT = "historical_1980_2019"              # must match Julia arg
STOCK_TYPE = "resstock"                           # or "comstock"
UPGRADE = 1                                       # an upgrade you processed
# -------------- #

parq_dir = PROJECT / f"data/load/{STOCK_TYPE}/simulated/bus_level"
files = sorted(parq_dir.glob(f"{EXPERIMENT}_{UPGRADE}_*.parquet"))
assert files, f"No parquet shards found in {parq_dir} for {EXPERIMENT} upgrade {UPGRADE}"

inputs_dir = PROJECT / "runs" / RUN_NAME / "inputs"
inputs_dir.mkdir(parents=True, exist_ok=True)
out_csv = inputs_dir / f"load_{EXPERIMENT}.csv"

# Helpers
def pick_time_col(df):
    cand = [c for c in df.columns if c.lower() in
            {"time","timestamp","datetime","date_time","ts"}]
    if cand:
        return cand[0]
    # heuristic: any datetime-like column?
    for c in df.columns:
        try:
            _ = pd.to_datetime(df[c], errors="raise")
            return c
        except Exception:
            pass
    raise AssertionError("Could not find a time-like column")

def pick_bus_col(df):
    # common bus labels
    names = {"bus","bus_id","bus_i","busnum","bus_number","node","zone","bus_name"}
    cand = [c for c in df.columns if c.lower() in names]
    if cand:
        return cand[0]
    # heuristic: columns that look like integer ids with low cardinality
    intish = [c for c in df.columns
              if pd.api.types.is_integer_dtype(df[c]) or
                 (pd.api.types.is_numeric_dtype(df[c]) and
                  np.all(np.mod(df[c].dropna().values,1)==0))]
    for c in intish:
        if df[c].nunique() > 1 and df[c].nunique() < 50000:
            return c
    raise AssertionError("Could not find a bus/node id column")

def pick_load_col(df):
    # Ordered preferences (lowercased); second element is MW scale
    prefs = [
        ("load_mw", 1.0),
        ("demand_mw", 1.0),
        ("savings_mw", 1.0),
        ("predicted_savings_mw", 1.0),
        ("mw", 1.0),
        ("value_mw", 1.0),
        ("load", 1.0),
        ("savings_kw", 1/1000.0),
        ("kw", 1/1000.0),
        ("value_kw", 1/1000.0),
    ]
    lc = {c.lower(): c for c in df.columns}

    for name, scale in prefs:
        if name in lc:
            return lc[name], scale

    # Heuristic: any numeric column with "mw" in the name
    mwish = [c for c in df.columns if "mw" in c.lower()
             and pd.api.types.is_numeric_dtype(df[c])]
    if len(mwish) == 1:
        return mwish[0], 1.0

    # Heuristic: any numeric column with "kw" in the name
    kwish = [c for c in df.columns if "kw" in c.lower()
             and pd.api.types.is_numeric_dtype(df[c])]
    if len(kwish) == 1:
        return kwish[0], 1/1000.0

    # Last resort: if there is a single numeric, non-id column left, use it
    nonmeta = [c for c in df.columns if c.lower() not in {"time","timestamp","datetime","date_time","ts",
                                                          "bus","bus_id","bus_i","busnum","bus_number","node","zone","bus_name"}]
    numeric = [c for c in nonmeta if pd.api.types.is_numeric_dtype(df[c])]
    if len(numeric) == 1:
        return numeric[0], 1.0

    # If we get here, print a helpful diagnostic and fail
    print("\n--- Column inspection (first shard) ---")
    print("Columns:", list(df.columns))
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    print("Numeric columns:", num_cols[:30])
    raise AssertionError("Could not identify a load column; see diagnostics above.")

dfs = []
for ix, fp in enumerate(files):
    df = pd.read_parquet(fp)
    # normalize column names for matching, but keep originals for output mapping
    # (we operate on the original df to preserve dtypes)
    tcol = pick_time_col(df)
    bcol = pick_bus_col(df)
    lcol, lscale = pick_load_col(df)

    if ix == 0:
        print(f"Detected columns from {fp.name}:")
        print(f"  time -> {tcol}")
        print(f"  bus  -> {bcol}")
        print(f"  load -> {lcol} (scale {lscale} to MW)")

    # time handling
    t = pd.to_datetime(df[tcol], utc=True, errors="coerce")
    if t.isna().any():
        # try localize-then-convert if naive timestamps
        t = pd.to_datetime(df[tcol], errors="coerce")
        t = t.dt.tz_localize("UTC", nonexistent="NaT", ambiguous="NaT")
    df2 = pd.DataFrame({
        "time": t,
        "bus": df[bcol],
        "load_MW": df[lcol] * lscale
    }).dropna(subset=["time","bus","load_MW"])

    # enforce types
    # (bus may be string or int; both are fine as long as consistent)
    dfs.append(df2)

# aggregate across shards (home types), sort, write
out = (pd.concat(dfs, ignore_index=True)
         .groupby(["time","bus"], as_index=False)["load_MW"].sum()
         .sort_values(["time","bus"]))

out.to_csv(out_csv, index=False)
print("\nWrote:", out_csv)
print("Rows:", len(out))
print("Time range:", out["time"].min(), "→", out["time"].max())
print("Unique buses:", out["bus"].nunique())


Detected columns from historical_1980_2019_1_mobile_home.parquet:
  time -> time
  bus  -> bus_id
  load -> bus_load_MW (scale 1.0 to MW)

Wrote: /home/fs01/jl2966/acorn-julia/runs/low_RE_high_elec_iter0/inputs/load_historical_1980_2019.csv
Rows: 10869840
Time range: 1980-01-01 01:00:00+00:00 → 2020-01-01 00:00:00+00:00
Unique buses: 31
